# ES Audit — Public & Private Collection Demo

Demonstrates two approaches on the live review environment:
- **Approach A** — Public collection: items indexed to the shared `dynastore-items` ES alias, searchable without authentication
- **Approach B** — Private collection: items go to the per-catalog ES only, STAC search returns nothing for anonymous callers

Covers fixes from PR #408:
| Fix | Verified by |
|-----|-------------|
| Items SEARCH routing ES→PG | Approach A — A.4 |
| OUTBOX alias maintenance | Approach A — A.5 |
| `is_private` ES field | Approach B — B.2/B.3 |
| Privacy apply-handler sync | Approach B — B.2 |
| STAC error surfacing | Approach B — B.6 |
| Registry re-key | Step 0 |

Run from the JupyterLite instance at `https://data.review.fao.org/geospatial/v2/api/catalog/web/#notebooks`.
`DYNASTORE_TOKEN` is optional — write operations (catalog/collection/config) require auth.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "https://data.review.fao.org/geospatial/v2/api/catalog")
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:6])

PUB_CATALOG_ID  = f"audit-pub-{RUN_ID}"
PRIV_CATALOG_ID = f"audit-priv-{RUN_ID}"
PUB_COLL_ID     = f"pub-coll-{RUN_ID}"
PRIV_COLL_ID    = f"priv-coll-{RUN_ID}"

# Content-Type must always be set so FastAPI decodes the body as JSON (not bytes).
default_headers = {"Content-Type": "application/json"}
if TOKEN:
    default_headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=default_headers, timeout=60.0)
# Anonymous client: Content-Type only, no auth — used for privacy probes.
anon   = httpx.Client(base_url=BASE_URL, headers={"Content-Type": "application/json"}, timeout=60.0)

print(f"Base URL : {BASE_URL}")
print(f"Run ID   : {RUN_ID}")
print(f"Auth     : {'token set ✓' if TOKEN else 'anonymous — write ops will 401'}")


## Step 0 — Verify registry re-key fix

GET `/configs/.../plugins/items_postgresql_driver` must return 200 (was 404 before the fix).

In [ ]:
# Create the public catalog first so we have a scope to probe.
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PUB_CATALOG_ID,
    "type": "Catalog", "stac_version": "1.0.0",
    "title": f"ES Audit public {RUN_ID}",
    "description": "Ephemeral — safe to delete",
    "links": [],
}))
print(f"Catalog {PUB_CATALOG_ID!r}: {r.status_code}")
if r.status_code not in (200, 201, 409):
    print(f"  body: {r.text[:200]}")

r = client.get(f"/configs/catalogs/{PUB_CATALOG_ID}/plugins/items_postgresql_driver")
print(f"GET items_postgresql_driver: {r.status_code}")
if r.status_code == 200:
    print("  ✓  registry re-key fix ACTIVE")
elif r.status_code == 404:
    print("  ✗  404 — PR #408 not yet deployed")
elif r.status_code == 401:
    print("  -  401 — need auth token to read configs")
else:
    print(f"  ?  {r.status_code}: {r.text[:150]}")


---
## Approach A — Public collection

Ingest items → OUTBOX drains to ES → alias covers new catalog → anonymous search succeeds.

### A.1 — Create public collection

In [ ]:
r = client.post(f"/stac/catalogs/{PUB_CATALOG_ID}/collections", content=json.dumps({
    "id": PUB_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Public items demo", "links": [], "title": "Public Items",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "license": "proprietary",
}))
print(f"Collection {PUB_COLL_ID!r}: {r.status_code}")
if r.status_code not in (200, 201, 409):
    print(f"  body: {r.text[:200]}")


### A.2 — Ingest 3 public STAC items

In [ ]:
PUB_ITEM_IDS = []
for i in range(3):
    item_id = f"pub-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PUB_CATALOG_ID}/collections/{PUB_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0",
            "id": item_id,
            "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.1, 41.9]},
            "bbox": [12.4 + i * 0.1, 41.8, 12.6 + i * 0.1, 42.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z"},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PUB_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code} ✓")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:120]}")

print(f"\nIndexed {len(PUB_ITEM_IDS)}/3 items. Waiting 10 s for OUTBOX drain …")
time.sleep(10)


### A.3 — Verify HATEOAS links use PUT not PATCH

In [ ]:
r = client.get(f"/configs/catalogs/{PUB_CATALOG_ID}/plugins/items_elasticsearch_driver")
print(f"GET items_elasticsearch_driver: {r.status_code}")
if r.status_code == 200:
    for lk in r.json().get("_links", []):
        rel    = lk.get("rel", "?")
        method = lk.get("method", "?").upper()
        flag   = "✗ STALE" if method == "PATCH" else "✓"
        print(f"  {flag}  rel={rel!r:20} method={method}")
elif r.status_code == 401:
    print("  -  401 — need auth token (skip)")
else:
    print(f"  (driver not registered — skip)")


### A.4 — STAC search returns items (ES routing fix)

Uses the authenticated client — same result expected for any caller on a public collection.

In [ ]:
r = client.post(
    f"/search/catalogs/{PUB_CATALOG_ID}/items-search",
    content=json.dumps({"limit": 10}),
)
print(f"POST /search/…/items-search: {r.status_code}")
if r.ok:
    feats = r.json().get("features", [])
    total = r.json().get("features_count", r.json().get("numberMatched", "?"))
    print(f"  features: {len(feats)} (total≈{total})")
    for f in feats[:3]:
        print(f"    {f['id']}")
    print("  ✓  ES SEARCH routing fix ACTIVE" if feats else "  ✗  0 features — routing fix not deployed or OUTBOX pending")
else:
    print(f"  {r.status_code}: {r.text[:300]}")


### A.5 — Anonymous search also returns items (public collection)

Public items must be visible without a Bearer token.

In [ ]:
r = anon.post(
    f"/search/catalogs/{PUB_CATALOG_ID}/items-search",
    content=json.dumps({"limit": 10}),
    headers={"Content-Type": "application/json"},
)
print(f"anonymous POST /search/…/items-search: {r.status_code}")
feats = r.json().get("features", []) if r.ok else []
print(f"  features: {len(feats)}")
if feats:
    print("  ✓  public items visible without auth")
else:
    print("  ✗  no items for anonymous — check routing + OUTBOX drain")


---
## Approach B — Private collection

`is_private=true` on a collection prevents anonymous access and denormalises into ES for privacy-aware queries.

### B.1 — Create private catalog + collection

In [ ]:
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PRIV_CATALOG_ID,
    "type": "Catalog", "stac_version": "1.0.0",
    "title": f"ES Audit private {RUN_ID}",
    "description": "Ephemeral — safe to delete",
    "links": [],
}))
print(f"Catalog {PRIV_CATALOG_ID!r}: {r.status_code}")

r = client.post(f"/stac/catalogs/{PRIV_CATALOG_ID}/collections", content=json.dumps({
    "id": PRIV_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Private items demo", "links": [], "title": "Private Items",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "license": "proprietary",
}))
print(f"Collection {PRIV_COLL_ID!r}: {r.status_code}")


### B.2 — Mark the collection private

Sets `is_private=true` via the `collection_privacy` config plugin. The apply-handler (commit `4c2b147`) syncs this to the ES collection doc.

In [ ]:
r = client.put(
    f"/configs/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/plugins/collection_privacy",
    content=json.dumps({"is_private": True}),
)
print(f"PUT collection_privacy: {r.status_code}")
if r.ok:
    print(f"  value: {r.json()}")
elif r.status_code == 401:
    print("  -  401 — need auth token to write configs")
else:
    print(f"  {r.text[:200]}")


### B.3 — Ingest 3 private items

In [ ]:
PRIV_ITEM_IDS = []
for i in range(3):
    item_id = f"priv-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0",
            "id": item_id,
            "geometry": {"type": "Point", "coordinates": [13.5 + i * 0.1, 42.9]},
            "bbox": [13.4 + i * 0.1, 42.8, 13.6 + i * 0.1, 43.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z", "confidential": True},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PRIV_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code} ✓")
    elif r.status_code == 500:
        print(f"  item {item_id}: 500 (generic — error surfacing fix not deployed)")
        try: print(f"    {r.json()}")
        except Exception: print(f"    {r.text[:200]}")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:120]}")

print(f"\nWaiting 10 s for OUTBOX drain …")
time.sleep(10)


### B.4 — Authenticated search returns private items

In [ ]:
if not TOKEN:
    print("(skipped — no token set)")
else:
    r = client.post(
        f"/search/catalogs/{PRIV_CATALOG_ID}/items-search",
        content=json.dumps({"limit": 10}),
    )
    print(f"authenticated POST /search/…/items-search: {r.status_code}")
    if r.ok:
        feats = r.json().get("features", [])
        print(f"  features: {len(feats)}")
        print("  ✓  private items visible to authenticated user" if feats else "  ✗  0 features — routing not resolving private ES driver")
    else:
        print(f"  {r.text[:300]}")


### B.5 — Anonymous search is blocked

The anonymous client (no `Authorization` header) must get 401/403 or zero features.

In [ ]:
r = anon.post(
    f"/search/catalogs/{PRIV_CATALOG_ID}/items-search",
    content=json.dumps({"limit": 10}),
    headers={"Content-Type": "application/json"},
)
print(f"anonymous POST /search/…/items-search: {r.status_code}")
feats = (r.json().get("features", []) if r.ok else [])
if r.status_code in (401, 403):
    print("  ✓  anonymous access rejected")
elif not feats:
    print("  ✓  anonymous returned 0 items (privacy enforced)")
else:
    print(f"  ✗  PRIVACY LEAK: anonymous got {len(feats)} items!")
    for f in feats[:2]:
        print(f"    {f.get('id')}")


### B.6 — STAC error surfacing: malformed item → real error code

Before the fix, all STAC errors were masked as generic 500.

In [ ]:
r = client.post(
    f"/stac/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/items",
    content=json.dumps({
        "type": "Feature", "stac_version": "1.0.0",
        "id": f"bad-{RUN_ID}",
        "geometry": "NOT_VALID",
        "bbox": [0],
        "properties": {},
        "links": [], "assets": {},
    }),
)
print(f"POST malformed item: {r.status_code}")
if r.status_code == 500:
    print("  ✗  generic 500 — error masking fix NOT deployed")
elif r.status_code in (400, 422):
    print(f"  ✓  validation error surfaced correctly: {r.status_code}")
    try: print(json.dumps(r.json(), indent=2)[:400])
    except Exception: print(r.text[:300])
elif r.status_code == 401:
    print("  -  401 — need auth to create items")
else:
    print(f"  {r.status_code} — {r.text[:200]}")


---
## Teardown

In [ ]:
for cat_id in [PUB_CATALOG_ID, PRIV_CATALOG_ID]:
    r = client.delete(f"/stac/catalogs/{cat_id}", params={"force": "true"})
    print(f"DELETE {cat_id!r}: {r.status_code}")
client.close()
anon.close()
print("Done.")
